## Data Quality Analysis (Missing, Duplicates and Data Types fixes)

In [314]:
import pandas as pd
import numpy as np
from configs.settings import project_dir

In [315]:
p_dir = project_dir()

#path to raw data files
raw_data_dir = p_dir.RAW_DATA_DIR
customers = raw_data_dir/'olist_customers_dataset.csv'
location = raw_data_dir/'olist_geolocation_dataset.csv'
items = raw_data_dir/'olist_order_items_dataset.csv'
payments = raw_data_dir/'olist_order_payments_dataset.csv'
reviews = raw_data_dir/'olist_order_reviews_dataset.csv'
orders = raw_data_dir/'olist_orders_dataset.csv'
products = raw_data_dir/'olist_products_dataset.csv'
sellers = raw_data_dir/'olist_sellers_dataset.csv'
category = raw_data_dir /'product_category_name_translation.csv'

### Customers Dataset 

In [316]:
customers_df = pd.read_csv(customers)
customers_df.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [317]:
# Missing Values
print(customers_df.isnull().sum())
print(f"\n\nNumber of rows with duplicates rows: {customers_df.duplicated().sum()}\n\n")
customers_df.info()

customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64


Number of rows with duplicates rows: 0


<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   customer_id               99441 non-null  str  
 1   customer_unique_id        99441 non-null  str  
 2   customer_zip_code_prefix  99441 non-null  int64
 3   customer_city             99441 non-null  str  
 4   customer_state            99441 non-null  str  
dtypes: int64(1), str(4)
memory usage: 3.8 MB


In [318]:
print(f"Number of duplicates value in customer_city col is {customers_df['customer_city'].unique().duplicated().sum()}\n")
print(f"Number of duplicates values in customer state is {customers_df['customer_state'].value_counts().duplicated().sum()}")

customers_df['customer_zip_code_prefix'].value_counts().sort_values(ascending=False).to_string('../arson/zip_code.txt')

Number of duplicates value in customer_city col is 0

Number of duplicates values in customer state is 0


### Geolocation Dataset

In [319]:
location_df = pd.read_csv(location)
location_df.head()

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP
3,1041,-23.544392,-46.639499,sao paulo,SP
4,1035,-23.541578,-46.641607,sao paulo,SP


In [320]:
location_df.shape

(1000163, 5)

In [321]:
# print(location_df.isnull().sum())
# print(f"Number of Duplicates in location dataset of {location_df['geolocation_city'].value_counts().sort_values().to_string('../arson/location_city.txt')}")

location_df['geolocation_city'] = location_df['geolocation_city'].astype('string')
location_df['geolocation_city'] = location_df['geolocation_city'].str.strip()
# location_df['geolocation_city'] = location_df['geolocation_city'].str.replace('`','',regex='False')
# location_df['geolocation_city'] = location_df['geolocation_city'].str.replace('^','',regex='False')
# location_df['geolocation_city'] = location_df['geolocation_city'].str.replace('~','',regex='False')
# location_df['geolocation_city'].value_counts().sort_values().to_string('../arson/location_city_clean.txt')

In [325]:
location_df[['geolocation_lat','geolocation_lng']].duplicated().sum()

np.int64(281700)

In [326]:
location_df['geolocation_city'].value_counts().sort_values().to_string('../arson/lat.csv')


In [327]:
location_df.head()
location_df.info()


<class 'pandas.DataFrame'>
RangeIndex: 1000163 entries, 0 to 1000162
Data columns (total 5 columns):
 #   Column                       Non-Null Count    Dtype  
---  ------                       --------------    -----  
 0   geolocation_zip_code_prefix  1000163 non-null  int64  
 1   geolocation_lat              1000163 non-null  float64
 2   geolocation_lng              1000163 non-null  float64
 3   geolocation_city             1000163 non-null  string 
 4   geolocation_state            1000163 non-null  str    
dtypes: float64(2), int64(1), str(1), string(1)
memory usage: 38.2 MB


In [328]:
messy_data = location_df[location_df[['geolocation_lat','geolocation_lng']].duplicated()]
# messy_data[['geolocation_city','geolocation_lat','geolocation_lng']].value_counts().head(30)
messy_data['geolocation_city'] = messy_data['geolocation_city'].astype(str)
messy_data['geolocation_city'] = messy_data['geolocation_city'].str.strip()
messy_data[['geolocation_lat','geolocation_lng','geolocation_city']].groupby(['geolocation_lat','geolocation_lng']).value_counts().sort_values(ascending=False).to_string('../arson/messy1.csv')

In [329]:
messy_data[['geolocation_lat','geolocation_lng']].duplicated()

15         False
44          True
65          True
66         False
67          True
           ...  
1000153     True
1000154     True
1000159    False
1000160    False
1000162     True
Length: 281700, dtype: bool

In [330]:
messy_data[messy_data['geolocation_city'] == 'xambre']

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
897964,87535,-23.729684,-53.489767,xambre,PR
898303,87535,-23.729684,-53.489767,xambre,PR
898430,87535,-23.733100,-53.488682,xambre,PR
898450,87535,-23.737584,-53.485975,xambre,PR


In [331]:
messy_data[['geolocation_city','geolocation_lat','geolocation_lng']][messy_data[['geolocation_lat','geolocation_lng']].duplicated()].head(30).sort_values(by=['geolocation_lat','geolocation_lng'])

,geolocation_city,geolocation_lat,geolocation_lng
237,sao paulo,-23.552496,-46.632060
337,sao paulo,-23.552235,-46.628441
136,sao paulo,-23.549854,-46.643139
161,sao paulo,-23.549854,-46.643139
223,sao paulo,-23.549854,-46.643139
240,sao paulo,-23.549854,-46.643139
280,sao paulo,-23.549819,-46.635606
253,sao paulo,-23.546935,-46.636588
275,sao paulo,-23.546935,-46.636588
306,são paulo,-23.546935,-46.636588


In [332]:
messy_data[['geolocation_lat','geolocation_lng','geolocation_city']].where(messy_data[['geolocation_lat','geolocation_lng']].duplicated())

,geolocation_lat,geolocation_lng,geolocation_city
15,NaN,NaN,NaN
44,-23.546081,-46.644820,sao paulo
65,-23.546081,-46.644820,sao paulo
66,NaN,NaN,NaN
67,-23.546081,-46.644820,sao paulo
...,...,...,...
1000153,-28.343273,-51.873734,ciriaco
1000154,-28.070493,-52.011342,tapejara
1000159,NaN,NaN,NaN
1000160,NaN,NaN,NaN


In [333]:
messy_data[messy_data['geolocation_zip_code_prefix'].duplicated()]

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
44,1046,-23.546081,-46.644820,sao paulo,SP
65,1046,-23.546081,-46.644820,sao paulo,SP
67,1046,-23.546081,-46.644820,sao paulo,SP
72,1046,-23.545320,-46.644069,sao paulo,SP
82,1046,-23.546081,-46.644820,sao paulo,SP
...,...,...,...,...,...
1000153,99970,-28.343273,-51.873734,ciriaco,RS
1000154,99950,-28.070493,-52.011342,tapejara,RS
1000159,99900,-27.877125,-52.224882,getulio vargas,RS
1000160,99950,-28.071855,-52.014716,tapejara,RS


In [334]:
location_df[location_df['geolocation_zip_code_prefix'].duplicated()].sort_values(by='geolocation_zip_code_prefix').head(10)

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
1384,1001,-23.549292,-46.633559,sao paulo,SP
206,1001,-23.550498,-46.634338,sao paulo,SP
1351,1001,-23.549951,-46.634027,são paulo,SP
235,1001,-23.550642,-46.634410,sao paulo,SP
985,1001,-23.550498,-46.634338,sao paulo,SP
1004,1001,-23.549292,-46.633559,sao paulo,SP
575,1001,-23.549779,-46.633957,são paulo,SP
519,1001,-23.551337,-46.634027,sao paulo,SP
1062,1001,-23.550498,-46.634338,sao paulo,SP
299,1001,-23.549698,-46.633909,sao paulo,SP


In [335]:
location_df_clean = location_df.drop_duplicates(subset=['geolocation_lat','geolocation_lng'])

In [340]:
print(location_df.shape)
print(location_df_clean.shape)

(1000163, 5)
(718463, 5)


In [341]:
1000163 - 718463

281700

In [339]:
location_df.duplicated().sum()

np.int64(261831)

In [338]:
location_df_clean.duplicated().sum()

np.int64(0)